In [140]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt 

In [141]:
words = open("names.txt", "r").read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [142]:
chars = sorted(list(set("".join(words))))

stoi = {s: i+1 for i, s in enumerate(chars)}
stoi["."] = 0 

itos = {i:s for s, i in stoi.items()}

In [143]:
# Build the dataset 
block_size = 3 
X, Y = [], [] 

for w in words:
   # print(w)

    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print("".join(itos[i] for i in context), "---->", itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [144]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [173]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [174]:
sum(p.nelement() for p in parameters) # total number of paramters 

3481

In [175]:
for p in parameters: 
    p.requires_grad = True 

In [176]:
# Since we have figured out the range where we think lies the optimal learning rate, we create a range of values between those two points 


# lr = torch.linspace(0.001, 1, steps=1000)

# This is a better way of re-implementing the code above 
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [177]:
lri = []
lossi = [] 

for i in range(1000):

    # mini-batch construct 
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass 
    # C[x[ix]] means we only want 32 rows of X which we will the be used to be indexed in C
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)


    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(32), Y].log().mean()

    # this one line of code is equivalent the the three commented lines above 
    # Y[ix] also select does same 32 rows corresponding Y label values 
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    # backward pass 
    for p in parameters:
        p.grad = None

    loss.backward() 
    # update 
    lr = lrs[i]

    for p in parameters: 
        p.data -= lr * p.grad 
    
    # track stats 
    lri.append(lrs)
    lossi.append(loss.item())


22.535036087036133
21.121341705322266
16.092300415039062
19.989830017089844
20.785240173339844
19.11100959777832
20.16458511352539
17.17803382873535
17.214765548706055
18.933727264404297
16.599008560180664
20.10748863220215
17.967113494873047
18.985591888427734
18.314756393432617
20.64729118347168
18.73198890686035
21.737882614135742
19.79482650756836
20.85912322998047
18.16280174255371
20.689382553100586
16.246511459350586
18.42610740661621
18.677490234375
18.76015281677246
17.71941375732422
20.075267791748047
19.172992706298828
19.806896209716797
19.529111862182617
18.00794219970703
18.57172203063965
18.999052047729492
15.613505363464355
19.121339797973633
20.088029861450195
15.592267036437988
19.45631980895996
17.308767318725586
18.34707260131836
20.37019920349121
16.906829833984375
16.5526065826416
19.86009979248047
16.666894912719727
15.985574722290039
16.385902404785156
18.211122512817383
17.862194061279297
16.994077682495117
20.01241683959961
20.214269638061523
15.12592029571533

In [149]:
# generate mini-batches 
# Select 32 training examples from X which will be 32 indexes 
torch.randint(0, X.shape[0], (32,))

tensor([153269,  32345, 166987, 171203,  71039, 209958, 161610,  40307,  74291,
         97334,  52263, 208644,  37551, 212050,  96533,  82957,  92691, 206628,
        169667, 173604, 122153, 160188, 135244,  65515,  42191, 144598, 196005,
         51581, 188801, 184191,  74071, 106565])